# Approach 1
This approach applies TCDB to acquire transporters and their TC identification (TCID). Subsequently, mechanisms for the relevant families are obtained manually, before they are connected to the reaction of the mechanism. This will in turn be crosschecked with Rhea, which is mapped to through UniProt IDs (UID).

Semantically, it will follow something along these lines: TCID + substrate + mechanism -> chemical reaction. Connect this and comparte with reaction on Rhea.

In [1]:
import requests
from Bio import SeqIO
from io import StringIO
import pandas as pd

The relevant files are as following:
1) "All proteins in TCDB (FASTA format)" - tc_fasta_url
2) "Tab-delimited table mapping TC systems to their substrates and ChEBI IDs" - tc_substrates_url

"Tab-delimited table mapping TC uniprot/refseq accessions to TC systems" is not necessary, as the UID is listed in the FASTA format file

In [2]:
def fetch_data(url):
    response = requests.get(url)
    response.raise_for_status()
    return response.text


def parse_data(fasta_txt, substrates_txt):
    # FASTA (TCID and UID)
    fasta_data = [[record.description.split("|")[3].split()[0], record.description.split("|")[2]]
                  for record in SeqIO.parse(StringIO(fasta_txt), "fasta")]

    df_fasta = pd.DataFrame(fasta_data, columns=["TCID", "UID"])

    # Substrates (TCID, CHEBI ID and CHEBI Name)
    substrates_lines = substrates_txt.strip().split("\n")
    substrate_data = [[line.split("\t")[0], chebi.split(";")[0], chebi.split(";")[1]]
        for line in substrates_lines
        for chebi in line.split("\t")[1].split("|")]
    
    df_substrates = pd.DataFrame(substrate_data, columns=["TCID", "CHEBI ID", "CHEBI Name"])


    return df_fasta, df_substrates

In [3]:
tc_fasta_url = "https://www.tcdb.org/public/tcdb"
tc_substrates_url = "https://www.tcdb.org/cgi-bin/substrates/getSubstrates.py"

tc_fasta_txt = fetch_data(tc_fasta_url)
tc_substrates_txt = fetch_data(tc_substrates_url)

df_fasta, df_substrates = parse_data(tc_fasta_txt, tc_substrates_txt)

In [5]:
df_substrates

,TCID,CHEBI ID,CHEBI Name
0,1.A.7.2.1,CHEBI:3308,calcium(2+)
1,3.A.3.1.8,CHEBI:9175,sodium(1+)
2,3.A.3.1.8,CHEBI:8345,potassium(1+)
3,2.A.6.2.48,CHEBI:6578,luteolin
4,2.A.6.2.48,CHEBI:3764,clotrimazole
...,...,...,...
14270,1.B.13.1.4,CHEBI:25367,molecule
14271,9.A.57.1.3,CHEBI:3308,calcium(2+)
14272,4.D.1.1.18,CHEBI:9823,UDP-N-acetyl-alpha-D-glucosamine
14273,1.A.8.11.6,CHEBI:5585,water


First I'll merge the DFs, before I reduce it, as only the top ten families in each of the subclasses below are of interest. These, alongside their general mechanism and acting entity can be obtained from Misc/All_comp/family_mechanisms_entity_all.tsv

The relevant subclasses were retrieved in Misc/TCDB_composition.ipynb, and are: [1.A, 1.B, 1.C, 2.A, 3.A]\
From these subclasses, the ten most populated familes were obtained for further analysis.

In [36]:
df = pd.merge(df_fasta, df_substrates, on="TCID", how="left")

# Importing the families and related mechanisms
df_family_mechanisms = pd.read_csv("../Misc/All_comp/families_mechanisms_entity_all.tsv", sep="\t")
df_family_mechanisms["Mechanism"] = df_family_mechanisms["Mechanism"].replace({"â‡Œ": "⇌", "â†’": "→"}, regex=True)

# Filter out families not in top 10 of each of the selected subclasses
df["Family"] = df["TCID"].apply(lambda x: ".".join(x.split(".")[:3]))
df = df[df["Family"].isin(df_family_mechanisms["Family"])]
df = df.merge(df_family_mechanisms[["Family", "Mechanism", "Acting Entity"]], on="Family", how="left")
df = df.drop(columns=["Family"])

Now, many of the CHEBI IDs are secondary IDs, and needs to be converted in order to map to Rhea for cross-checking.

In [37]:
df_s2p = pd.read_csv("../ChEBI/s2p.tsv", sep="\t")
secondary_to_primary = dict(zip(df_s2p["Secondary_ID"], df_s2p["Primary_ID"]))
df["CHEBI ID"] = df["CHEBI ID"].apply(lambda x: secondary_to_primary.get(x, x))

Future plan: Go through families_mechanisms_all.tsv and create another column for the acting entity in the reaction. The acting entity x will be marked as such: {x}\
This was first conudcted through AI, as this is a tedious manual task, before it was verified and edited by hand.

In [38]:
def create_reaction_row(row):

    if pd.isna(row["Acting Entity"]) or pd.isna(row["CHEBI Name"]):
        return row["Mechanism"]
    

    mechanisms = row["Mechanism"].split(", ")
    acting_entities = str(row["Acting Entity"]).split(", ")
    chebi_name = str(row["CHEBI Name"])

    reactions = []
    for mechanism, entity in zip(mechanisms, acting_entities):
        reaction = mechanism.replace(entity, chebi_name)
        reactions.append(reaction)
    
    return ", ".join(reactions)

df["Reaction"] = df.apply(create_reaction_row, axis=1)

Alright, the case is. There are some minor issues with the reactions as of now, but that is mainly when there are two active entities in the reaction (e.g. Me1/Me2). However, this will be an issue to look into later, and now the next focus will be to obtain the Rhea reactions, including both the names and the ChEBI IDs. This will go through UniProt.\
Starting off with mapping the correct Rhea IDs (RID) through a UniProt SPARQLE-query, saved as UniProt/UID_Rhea_mapping.tsv

In [39]:
# Excluding the direction of the reaction for now
uid_rhea_map = pd.read_csv("../UniProt/UID_Rhea_mapping.tsv", sep="\t", usecols=[0,1])

df = df.merge(uid_rhea_map, left_on="UID", right_on="protein", how="left")
df = df.rename(columns={"reaction": "RID"})
df["RID"] = df["RID"].apply(lambda x: f"RHEA:{int(x)}" if pd.notnull(x) else x)

The following step is to include the mapped reaction data for each RID. Both the equation, the ChEBI IDs and the ChEBI Name. This is stored in the columns titled as such: R:{col_name}\
The Rhea data is obtained from rhea-db.org 27.01.25, and can be found in the Rhea-folder.

In [40]:
rhea = pd.read_csv("../Rhea/Rhea.tsv", sep="\t")

rhea["Reaction identifier"] = rhea["Reaction identifier"].astype(str)
df["RID"] = df["RID"].astype(str)
df = df.merge(rhea, left_on="RID", right_on="Reaction identifier", how="left")

df = df.drop(columns=["protein", "Reaction identifier"])
df = df.rename(columns=lambda x: f"R:{x}" if x not in
               ["TCID", "UID", "CHEBI ID", "CHEBI Name", "Mechanism", "Acting Entity", "Reaction", "RID"]
               else x)

df

,TCID,UID,CHEBI ID,CHEBI Name,Mechanism,Acting Entity,Reaction,RID,R:Equation,R:ChEBI name,R:ChEBI identifier,R:EC number
0,3.A.1.12.16,5IIP_A,CHEBI:15354,choline,"Solute (in) + ATP → Solute (out) + ADP + Pi, S...","Solute, Substrate","choline (in) + ATP → choline (out) + ADP + Pi,...",nan,NaN,NaN,NaN,NaN
1,3.A.1.12.16,5IIP_A,CHEBI:3424,carnitinium,"Solute (in) + ATP → Solute (out) + ADP + Pi, S...","Solute, Substrate",carnitinium (in) + ATP → carnitinium (out) + A...,nan,NaN,NaN,NaN,NaN
2,3.A.1.12.16,5IIP_A,CHEBI:17750,glycine betaine,"Solute (in) + ATP → Solute (out) + ADP + Pi, S...","Solute, Substrate",glycine betaine (in) + ATP → glycine betaine (...,nan,NaN,NaN,NaN,NaN
3,3.A.1.12.16,5IIP_A,CHEBI:17203,L-proline,"Solute (in) + ATP → Solute (out) + ADP + Pi, S...","Solute, Substrate",L-proline (in) + ATP → L-proline (out) + ADP +...,nan,NaN,NaN,NaN,NaN
4,1.A.9.5.14,5O8F_E,CHEBI:17996,chloride,ions (in) ⇌ ions (out),ions,chloride (in) ⇌ chloride (out),nan,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
18588,1.A.17.5.18,XP_723491.2,NaN,NaN,"Cl- (out) ⇌ Cl- (in), Cations (out) ⇌ Cations ...","Cl-, Cations","Cl- (out) ⇌ Cl- (in), Cations (out) ⇌ Cations ...",nan,NaN,NaN,NaN,NaN
18589,3.A.1.211.22,YP_009001498.1,NaN,NaN,"Solute (in) + ATP → Solute (out) + ADP + Pi, S...","Solute, Substrate","Solute (in) + ATP → Solute (out) + ADP + Pi, S...",nan,NaN,NaN,NaN,NaN
18590,2.A.7.1.17,YP_009173407.1,NaN,NaN,Nan,NaN,Nan,nan,NaN,NaN,NaN,NaN
18591,1.A.1.7.8,YP_009174599.1,CHEBI:29103,potassium(1+),cation (out) ⇌ cation (in),cation,potassium(1+) (out) ⇌ potassium(1+) (in),nan,NaN,NaN,NaN,NaN


VERY many of the UIDs are refseq IDs (RSIDs), and hence unattainable through UniProt-Rhea mapping. Therefore, a query was written, and can be found in UniProt/UniProt_Rhea.ipynb as query2. This obtains the UID, RID, Rhea Reaction Direction (RDIR) and RSID whenever attainable. The query was run online on https://sparql.uniprot.org/, and the result is stored in query2.csv. After a second of thought, I realize that this file is too large to push to Git, so this must be created manually. UniProt/CSV_modification.ipynb modifies the csv as wanted, easing the use here. The same goes for the modified file, uid_rhea_refseq.tsv is close to the max push size, hence not included in the repo.

In [41]:
refseq = pd.read_csv("../UniProt/uid_rhea_refseq.tsv", sep="\t", usecols=[0,1,3])

df = df.merge(refseq, on='UID', how='left', suffixes=('', '_refseq'))
# df['RID'] = df['RID'].fillna(df['RID_refseq'])
# df.drop(columns=['RID_refseq'], inplace=True)
df

,TCID,UID,CHEBI ID,CHEBI Name,Mechanism,Acting Entity,Reaction,RID,R:Equation,R:ChEBI name,R:ChEBI identifier,R:EC number,RID_refseq,RSID
0,3.A.1.12.16,5IIP_A,CHEBI:15354,choline,"Solute (in) + ATP → Solute (out) + ADP + Pi, S...","Solute, Substrate","choline (in) + ATP → choline (out) + ADP + Pi,...",nan,NaN,NaN,NaN,NaN,NaN,NaN
1,3.A.1.12.16,5IIP_A,CHEBI:3424,carnitinium,"Solute (in) + ATP → Solute (out) + ADP + Pi, S...","Solute, Substrate",carnitinium (in) + ATP → carnitinium (out) + A...,nan,NaN,NaN,NaN,NaN,NaN,NaN
2,3.A.1.12.16,5IIP_A,CHEBI:17750,glycine betaine,"Solute (in) + ATP → Solute (out) + ADP + Pi, S...","Solute, Substrate",glycine betaine (in) + ATP → glycine betaine (...,nan,NaN,NaN,NaN,NaN,NaN,NaN
3,3.A.1.12.16,5IIP_A,CHEBI:17203,L-proline,"Solute (in) + ATP → Solute (out) + ADP + Pi, S...","Solute, Substrate",L-proline (in) + ATP → L-proline (out) + ADP +...,nan,NaN,NaN,NaN,NaN,NaN,NaN
4,1.A.9.5.14,5O8F_E,CHEBI:17996,chloride,ions (in) ⇌ ions (out),ions,chloride (in) ⇌ chloride (out),nan,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
254053,1.A.17.5.18,XP_723491.2,NaN,NaN,"Cl- (out) ⇌ Cl- (in), Cations (out) ⇌ Cations ...","Cl-, Cations","Cl- (out) ⇌ Cl- (in), Cations (out) ⇌ Cations ...",nan,NaN,NaN,NaN,NaN,NaN,NaN
254054,3.A.1.211.22,YP_009001498.1,NaN,NaN,"Solute (in) + ATP → Solute (out) + ADP + Pi, S...","Solute, Substrate","Solute (in) + ATP → Solute (out) + ADP + Pi, S...",nan,NaN,NaN,NaN,NaN,NaN,NaN
254055,2.A.7.1.17,YP_009173407.1,NaN,NaN,Nan,NaN,Nan,nan,NaN,NaN,NaN,NaN,NaN,NaN
254056,1.A.1.7.8,YP_009174599.1,CHEBI:29103,potassium(1+),cation (out) ⇌ cation (in),cation,potassium(1+) (out) ⇌ potassium(1+) (in),nan,NaN,NaN,NaN,NaN,NaN,NaN
